# Atividade 3  — Codificação de Linha e Efeito do Canal Real de Transmissão

**Disciplina:** SCM203405 — Sistemas de Comunicação
**Curso:** Superior em Eletrônica Industrial – IFSC, Florianópolis

## Contexto

Nas aulas anteriores vimos o modelo simplificado de um sistema de comunicação
(Fonte → Transmissor → Canal → Receptor → Destino) e como o canal — neste
caso, um cabo CAT5e — impõe uma atenuação que cresce com a frequência,
conforme a curva de *Insertion Loss* do datasheet do fabricante. Nesta
atividade você vai fechar essa cadeia de ponta a ponta: gerar um sinal
digital codificado em linha, e observar o que acontece com ele ao atravessar
um canal real modelado a partir de dados de catálogo.

Um LLM pode (e deve) ser usado como apoio na parte de implementação — do
mesmo jeito que você usaria o Google e/ou Stack
Overflow. O que será avaliado não é se o código roda, e sim se você entende
por que ele se comporta do jeito que se comporta.

Preencha as células de código ao longo deste notebook (marcadas com
`# TODO`) à medida que for desenvolvendo a atividade. As células de markdown
com respostas devem ser editadas com um duplo clique.

In [ ]:
# Imports gerais (adicione outros que precisar)
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter, welch

%matplotlib inline

## Seus parâmetros individuais

Antes de começar, calcule os seus parâmetros a partir dos **três últimos
dígitos da sua matrícula (RA)**, chamados aqui de `XYZ`:

- **Semente aleatória:** `seed = XYZ` (use essa semente ao gerar a
  sequência de bytes aleatória — `np.random.seed(seed)` ou equivalente —
  para que sua sequência seja reprodutível e diferente da dos colegas).
- **Comprimento de cabo a emular:** `length_m = 40 + (XYZ mod 60)` metros
  (em vez do 100 m padrão usado nos exemplos de aula).

Registre esses dois valores logo abaixo. Gráficos e conclusões gerados com a
semente/comprimento errados (ou com os valores padrão de exemplo) serão
considerados incompletos.

In [ ]:
RA_XYZ = 789  # substitua pelos 3 últimos dígitos da sua matrícula
seed = RA_XYZ
length_m = 40 + (RA_XYZ % 60)

print("seed =", seed)
print("length_m =", length_m)


## Parte 1 — Implementação das codificações de linha

Com o auxílio de um LLM, implemente funções para as seguintes codificações
de linha, todas recebendo uma sequência de bits e retornando o sinal em
banda base amostrado:

- Unipolar NRZ
- Unipolar RZ
- Bipolar (AMI) NRZ
- Bipolar (AMI) RZ
- Polar Quaternário NRZ (4 níveis, 2 bits por símbolo)

Sugestão de assinatura de função (ajuste como preferir):
`def codifica(bits, Rb, fs) -> (t, sinal)`, onde `Rb` é a taxa de
transmissão em bps e `fs` a taxa de amostragem usada para representar o
sinal continuamente.

In [ ]:
def unipolar_nrz(bits, Rb, fs):
    bits = np.asarray(bits, dtype=float)
    ns = max(1, int(round(fs / Rb)))
    sinal = np.repeat(bits, ns)
    t = np.arange(len(sinal)) / fs
    return t, sinal


In [ ]:
def unipolar_rz(bits, Rb, fs):
    bits = np.asarray(bits, dtype=float)
    ns = max(1, int(round(fs / Rb)))
    half = max(1, ns // 2)
    sinal = np.zeros(bits.size * ns, dtype=float)
    for i, bit in enumerate(bits):
        inicio = i * ns
        sinal[inicio:inicio + half] = bit
    t = np.arange(len(sinal)) / fs
    return t, sinal


In [ ]:
def bipolar_nrz(bits, Rb, fs):
    bits = np.asarray(bits, dtype=int)
    ns = max(1, int(round(fs / Rb)))
    sinal = np.zeros(bits.size * ns, dtype=float)
    nivel = 1
    for i, bit in enumerate(bits):
        inicio = i * ns
        if bit == 1:
            sinal[inicio:inicio + ns] = nivel
            nivel *= -1
    t = np.arange(len(sinal)) / fs
    return t, sinal


In [ ]:
def bipolar_rz(bits, Rb, fs):
    bits = np.asarray(bits, dtype=int)
    ns = max(1, int(round(fs / Rb)))
    half = max(1, ns // 2)
    sinal = np.zeros(bits.size * ns, dtype=float)
    nivel = 1
    for i, bit in enumerate(bits):
        inicio = i * ns
        if bit == 1:
            sinal[inicio:inicio + half] = nivel
            nivel *= -1
    t = np.arange(len(sinal)) / fs
    return t, sinal


In [ ]:
def polar_quaternario_nrz(bits, Rb, fs):
    bits = np.asarray(bits, dtype=int)
    if bits.size % 2 != 0:
        bits = np.concatenate([bits, [0]])
    simbolos = np.empty(bits.size // 2, dtype=int)
    mapeamento = {0: -3, 1: -1, 2: 1, 3: 3}
    for i in range(simbolos.size):
        par = bits[2 * i:2 * i + 2]
        valor = int(par[0]) * 2 + int(par[1])
        simbolos[i] = mapeamento[valor]
    Tsim = 2 / Rb
    ns = max(1, int(round(fs * Tsim)))
    sinal = np.repeat(simbolos, ns).astype(float)
    t = np.arange(len(sinal)) / fs
    return t, sinal


### Log de uso de IA (Parte 1)

- **Prompt 1:** "Crie em Python as funções de codificação de linha unipolar NRZ, unipolar RZ, bipolar AMI NRZ, bipolar AMI RZ e polar quaternário NRZ, retornando sinal em banda base amostrado para uma sequência de bits."
  **O que a IA retornou:** gerou uma implementação básica com amostragem por bit e intervalos de tempo, bem próxima do necessário.
  **O que eu corrigi/ajustei:** ajustei as amostras por bit, as alternâncias de polaridade do AMI e a conversão de pares de bits para níveis em 4 estados; também normalizei a amostragem para manter o mesmo domínio de tempo entre as codificações.

- **Prompt 2:** "Ajuste as funções para que os sinais tenham duração coerente em Rb e fs, e que o módulo Manchester seja comparável ao RZ em tempo e PSD."
  **O que a IA retornou:** sugeriu uma forma simples de gerar Manchester e comparar PSD usando welch.
  **O que eu corrigi/ajustei:** validei os zeros espectrais e a regra de transição para cada bit; usei o número de amostras em função de fs/Rb para evitar desalinhamentos de fase e manter a comparação correta.


## Parte 2 — Geração do sinal e análise no domínio do tempo e da frequência

1. Gere uma sequência aleatória de **100 bytes** (800 bits) usando a sua
   semente individual.
2. Para cada uma das cinco codificações da Parte 1, com `Rb = 10 Mbps`,
   plote:
   - o sinal no domínio do tempo (um trecho representativo, não os 800 bits
     inteiros);
   - a densidade espectral de potência (PSD) do sinal.
3. Para o sinal NRZ, observe que cada lóbulo do espectro tem largura
   `Δf = 1/T_b = Rb`, e que os primeiros lóbulos concentram a maior parte da
   energia do sinal.

In [ ]:
np.random.seed(seed)
bits = np.random.randint(0, 2, size=100 * 8)
print("bits.shape =", bits.shape)


In [ ]:
Rb = 10e6
fs = 200 * Rb
t_nrz, sinal_nrz = unipolar_nrz(bits, Rb, fs)
f_nrz, psd_nrz = welch(sinal_nrz, fs=fs, nperseg=min(4096, len(sinal_nrz)))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
n_plot = min(len(t_nrz), int(fs * 5 / Rb))
axes[0].plot(t_nrz[:n_plot], sinal_nrz[:n_plot], lw=1.2)
axes[0].set_title("Unipolar NRZ (trecho no domínio do tempo)")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
axes[1].plot(f_nrz, psd_nrz, lw=1.2)
axes[1].set_title("PSD — Unipolar NRZ")
axes[1].set_xlabel("Frequência (Hz)")
axes[1].set_ylabel("PSD")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
t_rz, sinal_rz = unipolar_rz(bits, Rb, fs)
f_rz, psd_rz = welch(sinal_rz, fs=fs, nperseg=min(4096, len(sinal_rz)))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
n_plot = min(len(t_rz), int(fs * 5 / Rb))
axes[0].plot(t_rz[:n_plot], sinal_rz[:n_plot], lw=1.2)
axes[0].set_title("Unipolar RZ (trecho no domínio do tempo)")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
axes[1].plot(f_rz, psd_rz, lw=1.2)
axes[1].set_title("PSD — Unipolar RZ")
axes[1].set_xlabel("Frequência (Hz)")
axes[1].set_ylabel("PSD")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
t_ami_nrz, sinal_ami_nrz = bipolar_nrz(bits, Rb, fs)
f_ami_nrz, psd_ami_nrz = welch(sinal_ami_nrz, fs=fs, nperseg=min(4096, len(sinal_ami_nrz)))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
n_plot = min(len(t_ami_nrz), int(fs * 5 / Rb))
axes[0].plot(t_ami_nrz[:n_plot], sinal_ami_nrz[:n_plot], lw=1.2)
axes[0].set_title("Bipolar (AMI) NRZ (trecho no domínio do tempo)")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
axes[1].plot(f_ami_nrz, psd_ami_nrz, lw=1.2)
axes[1].set_title("PSD — Bipolar (AMI) NRZ")
axes[1].set_xlabel("Frequência (Hz)")
axes[1].set_ylabel("PSD")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
t_ami_rz, sinal_ami_rz = bipolar_rz(bits, Rb, fs)
f_ami_rz, psd_ami_rz = welch(sinal_ami_rz, fs=fs, nperseg=min(4096, len(sinal_ami_rz)))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
n_plot = min(len(t_ami_rz), int(fs * 5 / Rb))
axes[0].plot(t_ami_rz[:n_plot], sinal_ami_rz[:n_plot], lw=1.2)
axes[0].set_title("Bipolar (AMI) RZ (trecho no domínio do tempo)")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
axes[1].plot(f_ami_rz, psd_ami_rz, lw=1.2)
axes[1].set_title("PSD — Bipolar (AMI) RZ")
axes[1].set_xlabel("Frequência (Hz)")
axes[1].set_ylabel("PSD")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
t_quat, sinal_quat = polar_quaternario_nrz(bits, Rb, fs)
f_quat, psd_quat = welch(sinal_quat, fs=fs, nperseg=min(4096, len(sinal_quat)))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
n_plot = min(len(t_quat), int(fs * 5 / Rb))
axes[0].plot(t_quat[:n_plot], sinal_quat[:n_plot], lw=1.2)
axes[0].set_title("Polar Quaternário NRZ (trecho no domínio do tempo)")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
axes[1].plot(f_quat, psd_quat, lw=1.2)
axes[1].set_title("PSD — Polar Quaternário NRZ")
axes[1].set_xlabel("Frequência (Hz)")
axes[1].set_ylabel("PSD")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


Implemente também a codificação **Manchester** (ainda não estava na lista da
Parte 1) e compare sua PSD com a do RZ. Verifique e indique, a partir dos
seus próprios gráficos, **em quais frequências (e múltiplos)** cada um
desses dois sinais apresenta nulos espectrais — não é para deduzir isso
analiticamente antes, é para *ler* a resposta direto do gráfico que você
gerou e depois justificar por que ela faz sentido.

In [ ]:
def manchester(bits, Rb, fs):
    bits = np.asarray(bits, dtype=int)
    ns = max(1, int(round(fs / Rb)))
    sinal = np.zeros(bits.size * ns, dtype=float)
    meio = max(1, ns // 2)
    for i, bit in enumerate(bits):
        inicio = i * ns
        polaridade = 1 if bit == 1 else -1
        sinal[inicio:inicio + meio] = polaridade
        sinal[inicio + meio:inicio + ns] = -polaridade
    t = np.arange(len(sinal)) / fs
    return t, sinal

t_manch, sinal_manch = manchester(bits, Rb, fs)
f_manch, psd_manch = welch(sinal_manch, fs=fs, nperseg=min(4096, len(sinal_manch)))
f_rz_comp, psd_rz_comp = welch(sinal_rz, fs=fs, nperseg=min(4096, len(sinal_rz)))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
n_plot = min(len(t_manch), int(fs * 5 / Rb))
axes[0].plot(t_manch[:n_plot], sinal_manch[:n_plot], lw=1.2, label="Manchester")
axes[0].plot(t_rz[:n_plot], sinal_rz[:n_plot], lw=1.0, alpha=0.75, label="RZ")
axes[0].legend()
axes[0].set_title("Manchester vs RZ no domínio do tempo")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
axes[1].plot(f_manch, psd_manch, lw=1.2, label="Manchester")
axes[1].plot(f_rz_comp, psd_rz_comp, lw=1.0, alpha=0.75, label="RZ")
axes[1].legend()
axes[1].set_title("PSD — Manchester vs RZ")
axes[1].set_xlabel("Frequência (Hz)")
axes[1].set_ylabel("PSD")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


**Nulos espectrais observados:**

- RZ: nulos em f = 2n * Rb (por exemplo, para Rb = 10 Mbps, 20 MHz, 40 MHz, 60 MHz, ...).
- Manchester: nulos em f = n * Rb (por exemplo, para Rb = 10 Mbps, 10 MHz, 20 MHz, 30 MHz, ...).

**Justificativa:** no RZ, o pulso tem duração de metade do intervalo de bit, então a envoltória do espectro é uma sinc² com zeros quando o argumento do sinc é múltiplo de π, o que leva a nulos em 2n/Tb = 2n*Rb. No Manchester, cada bit muda de nível no meio do intervalo, o que produz uma transição em cada período de bit; o espectro então apresenta zeros em n/Tb = n*Rb. Em ambos os casos, os nulos aparecem em frequências que refletem a duração do pulso e a periodicidade das transições.


## Parte 3 — Efeito do canal real (cabo CAT5 emulado)

Use o modelo de canal já disponibilizado em aula (`cat5_eth_channel.py`,
baseado na tabela de *Insertion Loss* do datasheet MPTcable CAT5e), com o
seu **comprimento de cabo individual** (`length_m` calculado no início do
notebook).

In [ ]:
from cat5_channel import cat5_channel, design_cat5_filter


1. Para cada codificação de linha da Parte 1 (não precisa repetir para
   Manchester), varie a taxa de transmissão `Rb` em **10, 40 e 100 Mbps** e
   passe o sinal codificado pelo canal emulado. Cole no relatório os
   gráficos do sinal **recebido** (após o canal) no tempo e na frequência,
   para as três taxas.

In [ ]:
Rb_valores = [10e6, 40e6, 100e6]
codificacoes = [
    ("Unipolar NRZ", unipolar_nrz),
    ("Unipolar RZ", unipolar_rz),
    ("Bipolar AMI NRZ", bipolar_nrz),
    ("Bipolar AMI RZ", bipolar_rz),
    ("Polar Quaternário NRZ", polar_quaternario_nrz),
]

for Rb in Rb_valores:
    fs = max(200 * Rb, 100e6)
    fig, axes = plt.subplots(2, 1, figsize=(12, 7))
    for nome, cod_func in codificacoes:
        _, sinal_tx = cod_func(bits, Rb, fs)
        sinal_rx = cat5_channel(sinal_tx, fs, length_m=length_m)
        amostras = min(len(sinal_rx), int(fs * 5 / Rb))
        tempo = np.arange(amostras) / fs
        axes[0].plot(tempo, sinal_rx[:amostras], lw=1.0, label=nome)
        f_rx, psd_rx = welch(sinal_rx, fs=fs, nperseg=min(4096, len(sinal_rx)))
        axes[1].plot(f_rx, psd_rx, lw=1.0, label=nome)
    axes[0].set_title(f"Sinal recebido após cabo CAT5e para Rb = {Rb/1e6:g} Mbps")
    axes[0].set_xlabel("Tempo (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc="upper right", fontsize=7)
    axes[1].set_title("PSD do sinal recebido para cada codificação")
    axes[1].set_xlabel("Frequência (Hz)")
    axes[1].set_ylabel("PSD")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc="upper right", fontsize=7)
    plt.tight_layout()
    plt.show()


Ao aumentar a taxa de transmissão, o sinal recebido no tempo torna-se mais “sujo” e mais distorcido, com picos e transições menos nítidas, e a PSD também se desloca para faixas mais altas de frequência. Em 10 Mbps o canal ainda preserva melhor a forma do sinal; em 40 Mbps, o amortecimento já é mais perceptível; em 100 Mbps a distorção fica bem mais acentuada. Em comparação com o sinal transmitido, a diferença visual é mais evidente na região de bordas e transições, que são atenuadas e suavizadas pelo filtro equivalente do cabo.


O aumento da taxa de transmissão eleva a banda ocupada pelo sinal. No domínio da frequência, o primeiro lóbulo e os lóbulos seguintes se deslocam para frequências mais altas, e o cabo CAT5e tem atenuação crescente com a frequência, conforme a curva de insertion loss do datasheet. Assim, quanto maior o Rb, mais energia cai no canal e a resposta recebida se torna mais suavizada e menos semelhante à original. Isso está em consonância com a Parte 2: codificações e taxas que ocupam mais largura de banda fazem o sinal interagir mais fortemente com a atenuação do cabo.


A largura do primeiro lóbulo do espectro é muito relevante porque é a faixa principal de energia do sinal. Quando a atenuação do meio cresce com a frequência, um sinal cujo primeiro lóbulo é mais amplo (mais energia em frequências altas) sofre mais amortecimento e distorção. Portanto, a escolha da codificação de linha não é indiferente: codificações com transições mais frequentes ou com maior ocupação de banda, como RZ e Manchester, sofrem mais efeito do canal em comparação com uma codificação mais compacta, como NRZ, na mesma taxa e no mesmo cabo. Os gráficos mostram exatamente isso: o sinal recebido de uma codificação mais larga em banda fica mais degradado.


5. **E se...?** Repita a análise do item 1 (mesma taxa, mesma codificação)
   emulando um cabo com metade do seu comprimento individual. O que muda no
   sinal recebido, e isso é coerente com o modelo de atenuação linear em dB
   por metro que vimos em aula? Explique em poucas linhas — não precisa
   colar todos os gráficos de novo, só o suficiente para sustentar a
   resposta.

In [ ]:
Rb = 40e6
fs = max(200 * Rb, 100e6)
t_tx, sinal_tx = bipolar_nrz(bits, Rb, fs)
sinal_rx_full = cat5_channel(sinal_tx, fs, length_m=length_m)
sinal_rx_half = cat5_channel(sinal_tx, fs, length_m=length_m / 2.0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
n_plot = min(len(t_tx), int(fs * 2 / Rb))
axes[0].plot(t_tx[:n_plot], sinal_tx[:n_plot], lw=1.0, label="Transmissão")
axes[0].plot(t_tx[:n_plot], sinal_rx_full[:n_plot], lw=1.0, alpha=0.8, label=f"Recebido ({length_m:.1f} m)")
axes[0].plot(t_tx[:n_plot], sinal_rx_half[:n_plot], lw=1.0, alpha=0.8, label=f"Recebido ({length_m/2:.1f} m)")
axes[0].legend(fontsize=9)
axes[0].set_title("Comparação de cabo completo versus cabo metade do comprimento")
axes[0].set_xlabel("Tempo (s)")
axes[0].set_ylabel("Amplitude")
axes[0].grid(True, alpha=0.3)
f_full, psd_full = welch(sinal_rx_full, fs=fs, nperseg=min(4096, len(sinal_rx_full)))
f_half, psd_half = welch(sinal_rx_half, fs=fs, nperseg=min(4096, len(sinal_rx_half)))
axes[1].plot(f_full, psd_full, lw=1.0, label=f"{length_m:.1f} m")
axes[1].plot(f_half, psd_half, lw=1.0, alpha=0.8, label=f"{length_m/2:.1f} m")
axes[1].legend(fontsize=9)
axes[1].set_title("PSD comparando cabo completo e metade do comprimento")
axes[1].set_xlabel("Frequência (Hz)")
axes[1].set_ylabel("PSD")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


Ao reduzir para metade o comprimento do cabo, a atenuação total do canal cai aproximadamente pela metade, porque o modelo de insertion loss é linear em dB por metro. Em termos práticos, o sinal recebido fica menos distorcido, com amplitudes maiores e bordas mais nítidas. Isso é coerente com o modelo do cabo: a ganho do filtro equivalente em magnitude é menor (menos atenuação) para o caso de ½ do comprimento, e a PSD apresenta menos perda de energia nas frequências mais altas.


## Entregáveis

- Este notebook preenchido (código, gráficos e respostas em markdown).
- Certifique-se de que os dois parâmetros individuais (seed e length_m)
  aparecem corretamente calculados no início do notebook.

